# MODULE 3 : Développement Applicatif
## II : Algorithmique et programmation appliquées


### Objectifs du jour
- Manipuler les structures de données adaptées à un cas métier (listes, dictionnaires)
- Écrire des fonctions robustes avec gestion d'erreurs
- Appliquer des règles de validation de données (un incontournable en entreprise)
- Construire les fonctions métier du projet fil rouge

### Déroulé de la journée
1. Rappel express : structures de données en contexte métier
2. Fonctions et gestion des erreurs (`try/except`)
3. Règles de validation des données
4. Atelier : fonctions métier du projet fil rouge
5. Exercice

---
## 1. Rappel express : structures de données en contexte métier

En entreprise, le choix de la structure de données n'est jamais anodin. Voici comment on raisonne :

| Besoin métier | Structure| Pourquoi |
|---|---|---|
| Une liste de clients | `list` | Ordre conservé, on peut avoir des doublons temporaires à filtrer |
| Retrouver un client par son ID rapidement | `dict` (`{id: client}`) | Accès direct en O(1), pas besoin de parcourir toute la liste |
| Les infos d'un client (nom, email...) | `dict` ou objet | Regrouper des données hétérogènes liées |
| Un ensemble d'emails déjà utilisés (pour éviter les doublons) | `set` | Vérification d'appartenance très rapide, pas de doublons possibles |

### Exemple 

In [25]:
clients = {
    1: {"nom": "Amina Traoré", "email": "amina@example.com", "telephone": "0612345678"},
    2: {"nom": "Jean Kouassi", "email": "jean.k@example.com", "telephone": "0623456789"},
}


for c in clients.values():
    print(c["email"])

amina@example.com
jean.k@example.com


In [31]:
animaux = ["chat", "chien", "chèvre", 2,"chat", False]
animaux

['chat', 'chien', 'chèvre', 2, 'chat', False]

In [32]:
animaux = {"chat", "chien", "chèvre", 2,"chat", False}
animaux

{2, False, 'chat', 'chien', 'chèvre'}

In [24]:
emails_utilises = {c["email"] for c in clients.values()}
emails_utilises

{'amina@example.com'}

In [33]:
print("Clients enregistrés :", clients)
print("Emails déjà utilisés :", emails_utilises)

Clients enregistrés : {1: {'nom': 'Amina Traoré', 'email': 'amina@example.com', 'telephone': '0612345678'}, 2: {'nom': 'Jean Kouassi', 'email': 'jean.k@example.com', 'telephone': '0623456789'}}
Emails déjà utilisés : {'amina@example.com'}


In [38]:
#Accès direct et rapide grâce au dictionnaire
print("\nAccès direct au client 2 :", clients[2])


Accès direct au client 2 : {'nom': 'Jean Kouassi', 'email': 'jean.k@example.com', 'telephone': '0623456789'}


---
## 2. Fonctions et gestion des erreurs

Un script plante quand les données sont mauvaises. **Une application professionnelle anticipe les erreurs et réagit proprement.**

### a) Rappel : structure d'une fonction propre

In [46]:
def calculer_total_commande(quantite, prix_unitaire):
    """
    Calcule le montant total d'une commande.

    Args:
        quantite (int): nombre d'unités commandées
        prix_unitaire (float): prix d'une unité

    Returns:
        float: montant total
    """
    return quantite * prix_unitaire


# Notez la docstring : en entreprise, TOUTE fonction publique doit être documentée.
# Un collègue (ou vous, dans 6 mois) doit comprendre la fonction sans lire son code.

print(calculer_total_commande(3, 15.5))

46.5


### b) Que se passe-t-il avec de mauvaises données ?

In [53]:
print(calculer_total_commande("trois", 15.5))

TypeError: can't multiply sequence by non-int of type 'float'

In [3]:
try:
    print(calculer_total_commande("trois", 15.5))
except TypeError as e:
    print(f"Erreur détectée : {e}")

# Sans le try/except, ce genre d'erreur ferait planter TOUTE l'application
# et l'utilisateur final ne comprendrait rien (mauvaise expérience, image dégradée de l'entreprise)

Erreur détectée : can't multiply sequence by non-int of type 'float'


### c) Version professionnelle : on valide AVANT de calculer

En entreprise, on préfère souvent **anticiper** l'erreur plutôt que la laisser se produire puis la rattraper. C'est plus lisible et plus sûr.

In [4]:
def calculer_total_commande_v2(quantite, prix_unitaire):
    """
    Calcule le montant total d'une commande, avec validation des entrées.

    Lève ValueError si les données sont invalides.
    """
    if not isinstance(quantite, (int, float)) or not isinstance(prix_unitaire, (int, float)):
        raise ValueError("La quantité et le prix doivent être des nombres.")
    if quantite <= 0:
        raise ValueError("La quantité doit être strictement positive.")
    if prix_unitaire < 0:
        raise ValueError("Le prix ne peut pas être négatif.")
    return round(quantite * prix_unitaire, 2)


# Tests
cas_de_test = [
    (3, 15.5),      # cas valide
    (0, 10),        # quantité nulle -> doit lever une erreur
    (-2, 10),       # quantité négative -> doit lever une erreur
    (2, -5),        # prix négatif -> doit lever une erreur
]

for quantite, prix in cas_de_test:
    try:
        resultat = calculer_total_commande_v2(quantite, prix)
        print(f"OK  -> quantite={quantite}, prix={prix} => total={resultat}")
    except ValueError as erreur:
        print(f"REJET -> quantite={quantite}, prix={prix} => {erreur}")

OK  -> quantite=3, prix=15.5 => total=46.5
REJET -> quantite=0, prix=10 => La quantité doit être strictement positive.
REJET -> quantite=-2, prix=10 => La quantité doit être strictement positive.
REJET -> quantite=2, prix=-5 => Le prix ne peut pas être négatif.


---
## 3. Règles de validation des données

Reprenons notre cahier des charges. Un email doit ressembler à un email, un téléphone à un téléphone. On utilise le module `re` (expressions régulières), très utilisé  pour la validation de formulaires.

In [59]:
import re

def valider_email(email):
    """Retourne True si le format de l'email est valide, False sinon."""
    motif = r"^[\w.\-]+@[\w\-]+\.[a-zA-Z]{2,}$"
    return re.match(motif, email) is not None


def valider_telephone(telephone):
    """Retourne True si le téléphone contient exactement 10 chiffres."""
    return telephone.isdigit() and len(telephone) == 10


# Tests
emails_test = ["amina@example.com", "pas-un-email", "jean.k@societe.fr", "@manque-partie.com"]
for email in emails_test:
    print(f"{email:40s} -> {'valide' if valider_email(email) else 'INVALIDE'}")

print()
telephones_test = ["0612345678", "061234", "06-12-34-56-78"]
for tel in telephones_test:
    print(f"{tel:20s} -> {'valide' if valider_telephone(tel) else 'INVALIDE'}")

amina@example.com                        -> valide
pas-un-email                             -> INVALIDE
jean.k@societe.fr                        -> valide
@manque-partie.com                       -> INVALIDE

0612345678           -> valide
061234               -> INVALIDE
06-12-34-56-78       -> INVALIDE


---
## 4. Fonctions métier du projet fil rouge

On construit maintenant, étape par étape, les fonctions métier qui serviront de base au reste de la semaine. On reste volontairement en **dictionnaires/listes** aujourd'hui : la conversion en **classes** (POO) aura lieu demain, pour bien distinguer "la logique" de "la structure objet".

In [6]:
# Notre "base de données" en mémoire pour aujourd'hui
clients = {}          # {id_client: {...}}
commandes = []         # liste de dictionnaires commande
prochain_id_client = 1
prochain_id_commande = 1


def ajouter_client(nom, email, telephone):
    """
    Ajoute un client après validation des données.
    Retourne l'id du client créé, ou lève ValueError si les données sont invalides.
    """
    global prochain_id_client

    if not nom or not nom.strip():
        raise ValueError("Le nom du client est obligatoire.")
    if not valider_email(email):
        raise ValueError(f"Email invalide : {email}")
    if not valider_telephone(telephone):
        raise ValueError(f"Téléphone invalide : {telephone} (10 chiffres attendus)")

    # Vérifie qu'on n'a pas déjà ce client (règle métier : pas de doublon d'email)
    for client in clients.values():
        if client["email"] == email:
            raise ValueError(f"Un client avec l'email {email} existe déjà.")

    id_client = prochain_id_client
    clients[id_client] = {"nom": nom.strip(), "email": email, "telephone": telephone}
    prochain_id_client += 1
    return id_client


# Test
id1 = ajouter_client("Amina Traoré", "amina@example.com", "0612345678")
id2 = ajouter_client("Jean Kouassi", "jean.k@example.com", "0623456789")
print("Clients créés :", clients)

try:
    ajouter_client("Doublon", "amina@example.com", "0699999999")
except ValueError as erreur:
    print(f"\nRejet attendu : {erreur}")

Clients créés : {1: {'nom': 'Amina Traoré', 'email': 'amina@example.com', 'telephone': '0612345678'}, 2: {'nom': 'Jean Kouassi', 'email': 'jean.k@example.com', 'telephone': '0623456789'}}

Rejet attendu : Un client avec l'email amina@example.com existe déjà.


In [7]:
def ajouter_commande(id_client, date, produit, quantite, prix_unitaire):
    """
    Ajoute une commande pour un client existant, après validation.
    Retourne l'id de la commande créée.
    """
    global prochain_id_commande

    # Règle métier essentielle : pas de commande sans client existant !
    if id_client not in clients:
        raise ValueError(f"Client inconnu (id={id_client}). Impossible de créer la commande.")

    total = calculer_total_commande_v2(quantite, prix_unitaire)  # réutilise la validation du point 2

    commande = {
        "id_commande": prochain_id_commande,
        "id_client": id_client,
        "date": date,
        "produit": produit,
        "quantite": quantite,
        "prix_unitaire": prix_unitaire,
        "total": total,
    }
    commandes.append(commande)
    prochain_id_commande += 1
    return commande["id_commande"]


def lister_commandes_client(id_client):
    """Retourne la liste des commandes d'un client donné."""
    return [c for c in commandes if c["id_client"] == id_client]


def total_depense_client(id_client):
    """Calcule le montant total dépensé par un client."""
    return sum(c["total"] for c in lister_commandes_client(id_client))


# Tests
ajouter_commande(id1, "2026-08-20", "Clavier mécanique", 2, 45.0)
ajouter_commande(id1, "2026-08-22", "Souris sans fil", 1, 25.0)
ajouter_commande(id2, "2026-08-21", "Écran 24 pouces", 1, 150.0)

print("Commandes du client 1 :", lister_commandes_client(id1))
print("\nTotal dépensé par le client 1 :", total_depense_client(id1), "€")
print("Total dépensé par le client 2 :", total_depense_client(id2), "€")

try:
    ajouter_commande(99, "2026-08-22", "Produit fantôme", 1, 10.0)
except ValueError as erreur:
    print(f"\nRejet attendu (client inconnu) : {erreur}")

Commandes du client 1 : [{'id_commande': 1, 'id_client': 1, 'date': '2026-08-20', 'produit': 'Clavier mécanique', 'quantite': 2, 'prix_unitaire': 45.0, 'total': 90.0}, {'id_commande': 2, 'id_client': 1, 'date': '2026-08-22', 'produit': 'Souris sans fil', 'quantite': 1, 'prix_unitaire': 25.0, 'total': 25.0}]

Total dépensé par le client 1 : 115.0 €
Total dépensé par le client 2 : 150.0 €

Rejet attendu (client inconnu) : Client inconnu (id=99). Impossible de créer la commande.


---
## 5. Exercice 

Le service commercial ajoute une nouvelle exigence :

> *"On aimerait aussi pouvoir appliquer une remise en pourcentage sur une commande, et connaître le client qui a le plus dépensé."*

### À faire :

1. Écrivez une fonction `appliquer_remise(id_commande, pourcentage)` qui :
   - vérifie que la commande existe (sinon lève `ValueError`)
   - vérifie que le pourcentage est entre 0 et 100 (sinon lève `ValueError`)
   - recalcule et met à jour le `total` de la commande
   - retourne le nouveau total

2. Écrivez une fonction `meilleur_client()` qui retourne l'id et le nom du client ayant le plus dépensé (utilisez `total_depense_client` pour chaque client)

3. Testez vos deux fonctions avec au moins un cas valide et un cas d'erreur chacune


In [ ]:
# EXERCICE 

def appliquer_remise(id_commande, pourcentage):
    """TODO : documentez et implémentez cette fonction."""
    pass


def meilleur_client():
    """TODO : documentez et implémentez cette fonction."""
    pass


# Vos tests ici
